# Error Case 11 — 403 Forbidden: Principal Role Not Assigned

**Error:** `403 Forbidden`  
**Root cause:** Principal exists, principal role exists, catalog role exists — but principal role NOT linked to principal  
**Fix:** Assign principal role to principal

## Flow
```
GET /catalog/v1/{catalog}/namespaces
  → Auth: JWT valid ✅
  → RBAC: find principal roles for principal
  → 0 principal roles → no grants possible
  → 403 ForbiddenException
```

## Difference from Notebook 04
```
Notebook 04: principal role assigned ✅
             catalog role NOT assigned to principal role

Notebook 11: catalog role assigned to principal role ✅
             principal role NOT assigned to principal  ← this case
```

In [1]:
# ── src path bootstrap (shared modules live in /src) ────────────────────────────
import sys, pathlib
_ROOT = pathlib.Path.cwd()
while not (_ROOT / "src").is_dir() and _ROOT != _ROOT.parent:
    _ROOT = _ROOT.parent
_SRC = _ROOT / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import sys
import importlib
import nb_support
importlib.reload(nb_support)
from nb_support import *
require_not_prod("error-case test")  # PROD-safe: refuses to run against company PROD

cleanup()
print('✅ Setup complete')

🧹 Cleaning up test resources...
✅ Cleanup done
✅ Setup complete


In [ ]:
# Cell 2: Get valid token first
r = get_token()
print_response(r, 'Valid token response')
valid_token = r.json()['access_token']
print(f'\n✅ Got valid token')

In [2]:
# Cell 2: Create full RBAC chain EXCEPT principal role → principal link
r1 = create_catalog()
assert r1.status_code == 201, r1.text

r2 = create_principal()
assert r2.status_code == 201
cred = r2.json()['credentials']

r3 = create_principal_role()
assert r3.status_code == 201

r4 = create_catalog_role()
assert r4.status_code == 201

r5 = grant_privilege(privilege='CATALOG_MANAGE_CONTENT')
assert r5.status_code == 201

r6 = assign_catalog_role_to_principal_role()
assert r6.status_code == 201

# ← intentionally skip: assign_principal_role_to_principal()

print('Setup:')
print('  catalog ✅')
print('  principal ✅')
print('  principal role ✅')
print('  catalog role → principal role ✅')
print('  principal role → principal ❌  ← missing link')

Setup:
  catalog ✅
  principal ✅
  principal role ✅
  catalog role → principal role ✅
  principal role → principal ❌  ← missing link


In [3]:
# Cell 3: Trigger 403 — principal has no roles
r_tok = get_token(cred['clientId'], cred['clientSecret'])
assert r_tok.status_code == 200
principal_token = r_tok.json()['access_token']

r = requests.get(
    f'{BASE_CAT}/{TEST_CATALOG}/namespaces',
    headers=h(principal_token)
)
print_response(r, '403 Principal role not assigned')
assert r.status_code == 403
print('\n✅ 403 confirmed — principal role not linked to principal')


403 Principal role not assigned:
  Status:     403
  Request-Id: 9887a791-517e-47dd-b249-e2806aed84a9_0000000000000000063
  Body: {
  "error": {
    "message": "Principal 'test-principal' with activated PrincipalRoles '[]' and activated grants via '[]' is not authorized for op LIST_NAMESPACES",
    "type": "ForbiddenException",
    "code": 403
  }
}

✅ 403 confirmed — principal role not linked to principal


In [4]:
# Cell 4: Find error logs
wait_for_logs(5)
print('Searching for 403 errors (last 1 min)...')
hits = search_403_errors(minutes_ago=1)
print_logs(hits)

⏳ Waiting 10s for logs to reach OpenSearch...
Searching for 403 errors (last 1 min)...
Found 10 log entries:

ℹ️  [2026-06-02T05:50:14.475Z] INFO
   logger    : io.quarkus.http.access-log
   requestId : 9887a791-517e-47dd-b249-e2806aed84a9_0000000000000000063
   realmId   : POLARIS
   message   : 192.168.194.1 - test-principal [02/Jun/2026:05:50:14 +0000] "GET /api/catalog/v1/test-error-catalog/namespaces HTTP/1.1" 403 195

🔍 [2026-06-02T05:50:14.474Z] DEBUG
   logger    : org.apache.polaris.service.config.DefaultConfigurationStore
   requestId : 9887a791-517e-47dd-b249-e2806aed84a9_0000000000000000063
   realmId   : POLARIS
   message   : Get configuration value for ENFORCE_PRINCIPAL_CREDENTIAL_ROTATION_REQUIRED_CHECKING with realm POLARIS

🔍 [2026-06-02T05:50:14.474Z] DEBUG
   logger    : org.apache.polaris.core.auth.PolarisAuthorizerImpl
   requestId : 9887a791-517e-47dd-b249-e2806aed84a9_0000000000000000063
   realmId   : POLARIS
   message   : Failed to satisfy privilege NAMESPACE

In [5]:
# Cell 5: Fix — assign principal role to principal
r_fix = assign_principal_role_to_principal()
print_response(r_fix, 'Assign principal role to principal')
assert r_fix.status_code == 201

# Verify — get fresh token after assignment
r_tok = get_token(cred['clientId'], cred['clientSecret'])
principal_token = r_tok.json()['access_token']

r_verify = requests.get(
    f'{BASE_CAT}/{TEST_CATALOG}/namespaces',
    headers=h(principal_token)
)
print(f'\nVerify: {r_verify.status_code}')
assert r_verify.status_code == 200
print('\n✅ Fixed — principal role assigned → access granted')

cleanup()
print('\n=== Summary ===')
print('Error:    403 Forbidden')
print('Cause:    Principal role exists but not assigned to principal')
print('Detect:   403 with ForbiddenException, 0 activated PrincipalRoles')
print('Fix:      PUT /principals/{name}/principal-roles')
print('Check:    GET /principals/{name}/principal-roles → should list roles')
print('Note:     Different from nb04 — RBAC chain broken at different link')


Assign principal role to principal:
  Status:     201
  Request-Id: 9887a791-517e-47dd-b249-e2806aed84a9_0000000000000000065
  Body: 

Verify: 200

✅ Fixed — principal role assigned → access granted
🧹 Cleaning up test resources...
✅ Cleanup done

=== Summary ===
Error:    403 Forbidden
Cause:    Principal role exists but not assigned to principal
Detect:   403 with ForbiddenException, 0 activated PrincipalRoles
Fix:      PUT /principals/{name}/principal-roles
Check:    GET /principals/{name}/principal-roles → should list roles
Note:     Different from nb04 — RBAC chain broken at different link
